# 06 · Final model: emotions and 5-day realised volatility

This notebook puts the paper's final specification (Section 5, Eq. 1) in one place:

$$\sigma_t = \alpha + \rho\,\sigma_{t-1} + \sum_{k=1}^{6} \delta_k\, S_{k,t} + \gamma' X_{t-1} + \varepsilon_t, \qquad S_{k,t} = \tfrac{1}{2}\,(S_{k,t-1} + S_{k,t-2})$$

- $\sigma_t$: 5-day realised volatility (rolling SD of daily log returns)
- $S_{k,t}$: two-day average of the daily score for emotion $k$ (joy, trust, fear, surprise, sadness, anger), so sentiment always comes before volatility
- $X_{t-1}$: log volume, log transaction count, log active addresses, Google Trends, Crypto Fear & Greed Index, Musk tweet dummy
- OLS with Newey–West (HAC) standard errors, 5 lags

**About the data.** The paper's emotion scores come from the final ATF prompt (`scripts/score_emotions_atf.py`) and are weighted by each post's upvotes. This repo ships the scores from the first full labelling pass (`data/Doge_Sentiment_Master.csv`) and no upvote counts. The model below is therefore the equal-weighted version on those scores. It is compared below with the paper's equal-weighted column (Table 3, column 3), the one result in the paper that uses the same weighting. The robustness and regime tests in the paper use the upvote-weighted ATF scores, so they are not repeated here.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

EMOTIONS = ['joy', 'tru', 'fea', 'sur', 'sad', 'ang']
NAMES = dict(joy='Joy', tru='Trust', fea='Fear', sur='Surprise', sad='Sadness', ang='Anger')
CONTROLS = ['log_volume', 'log_TxCnt', 'log_AdrActCnt', 'google_trends', 'fng_value', 'musk_dummy']

# post-level scores -> daily equal-weighted means
posts = pd.read_csv('../data/Doge_Sentiment_Master.csv', encoding='utf-8-sig')
posts['date'] = pd.to_datetime(posts['date'], dayfirst=True)
daily = posts.groupby('date')[EMOTIONS].mean()

market = pd.read_csv('../data/Master_Final.csv', parse_dates=['date']).set_index('date')
df = market.join(daily, how='left')
print(f'{len(posts):,} posts, {daily.index.nunique()} days with posts, {len(df)} trading days')

56,799 posts, 365 days with posts, 365 trading days


In [2]:
# regressors: two-day average of emotions (t-1, t-2), controls and lagged volatility at t-1
for k in EMOTIONS:
    df[f'{k}_c'] = (df[k].shift(1) + df[k].shift(2)) / 2
for c in CONTROLS:
    df[f'{c}_l1'] = df[c].shift(1)
df['vol_l1'] = df['volatility_5d'].shift(1)

rhs = ['vol_l1'] + [f'{k}_c' for k in EMOTIONS] + [f'{c}_l1' for c in CONTROLS]
reg = df.dropna(subset=['volatility_5d'] + rhs)
base = smf.ols('volatility_5d ~ ' + ' + '.join(rhs), data=reg).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
print(base.summary())

                            OLS Regression Results                            
Dep. Variable:          volatility_5d   R-squared:                       0.779
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     71.01
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           1.38e-89
Time:                        01:00:32   Log-Likelihood:                 1028.1
No. Observations:                 363   AIC:                            -2028.
Df Residuals:                     349   BIC:                            -1974.
Df Model:                          13                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -0.1510      0.050  

In [3]:
# side-by-side with the paper's equal-weighted column (Table 3, col. 3)
paper = {'vol_l1': 0.712, 'joy_c': 0.006, 'tru_c': 0.007, 'fea_c': 0.121, 'sur_c': 0.010,
         'sad_c': 0.088, 'ang_c': -0.018, 'log_volume_l1': 0.007, 'musk_dummy_l1': -0.007}

def stars(p): return '***' if p < .01 else '**' if p < .05 else '*' if p < .10 else ''
rows = []
for v, pv in paper.items():
    rows.append({'variable': v, 'paper': pv,
                 'this repo': f"{base.params[v]:.3f}{stars(base.pvalues[v])}",
                 'SE': round(base.bse[v], 3)})
cmp = pd.DataFrame(rows).set_index('variable')
print(cmp.to_string())
print(f"\nN = {int(base.nobs)} (paper 363) · R² = {base.rsquared:.3f} (paper 0.782)")

               paper this repo     SE
variable                             
vol_l1         0.712  0.712***  0.043
joy_c          0.006     0.009  0.037
tru_c          0.007     0.036  0.035
fea_c          0.121    0.117*  0.070
sur_c          0.010     0.013  0.039
sad_c          0.088     0.038  0.066
ang_c         -0.018     0.009  0.065
log_volume_l1  0.007  0.007***  0.002
musk_dummy_l1 -0.007  -0.007**  0.004

N = 363 (paper 363) · R² = 0.779 (paper 0.782)


## Hypothesis tests

One-sided Wald tests on the model above, following the paper's Table 4: does fear move volatility more than joy (across valence) and more than sadness (same valence, lower arousal)? The paper runs these on the upvote-weighted ATF scores; these numbers use the equal-weighted earlier scores.

In [4]:
def one_sided(model, expr, direction='>'):
    t = model.t_test(expr)
    est, se, tval = float(t.effect[0]), float(t.sd[0][0]), float(t.tvalue[0][0])
    p_two = float(t.pvalue)
    p = p_two / 2 if (tval > 0) == (direction == '>') else 1 - p_two / 2
    return est, se, tval, p

tests = [('Fear > Joy',     'fea_c - joy_c = 0', '>'),
         ('Fear > Sadness', 'fea_c - sad_c = 0', '>'),
         ('Joy < 0',        'joy_c = 0',         '<'),
         ('Trust < 0',      'tru_c = 0',         '<')]
out = pd.DataFrame([(name, *one_sided(base, e, d)) for name, e, d in tests],
                   columns=['H1 alternative', 'estimate', 'SE', 't', 'one-sided p']).round(3)
print(out.to_string(index=False))

H1 alternative  estimate    SE     t  one-sided p
    Fear > Joy     0.108 0.073 1.466        0.071
Fear > Sadness     0.079 0.112 0.709        0.239
       Joy < 0     0.009 0.037 0.252        0.600
     Trust < 0     0.036 0.035 1.023        0.847


## Takeaway

On these scores, fear is the only emotion with a significant coefficient (10% level) once lagged volatility, volume, on-chain activity and attention are controlled for. The fear–joy gap is significant at 10% (one-sided). The fear–sadness gap is positive but not significant here; in the paper, with the ATF prompt and upvote weighting, both gaps are significant at 5%.

The gap between this notebook and the paper comes from measurement: which prompt produced the scores, and how posts are weighted. Checking the LLM scores against a hand-labelled sample is the obvious next step.